In [2]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame

print(df.shape)
display(df.head())
display(df.describe().T)

(20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


,count,mean,std,min,25%,50%,75%,max
MedInc,20640.0,3.870671,1.899822,0.499900,2.563400,3.534800,4.743250,15.000100
HouseAge,20640.0,28.639486,12.585558,1.000000,18.000000,29.000000,37.000000,52.000000
AveRooms,20640.0,5.429000,2.474173,0.846154,4.440716,5.229129,6.052381,141.909091
AveBedrms,20640.0,1.096675,0.473911,0.333333,1.006079,1.048780,1.099526,34.066667
Population,20640.0,1425.476744,1132.462122,3.000000,787.000000,1166.000000,1725.000000,35682.000000
AveOccup,20640.0,3.070655,10.386050,0.692308,2.429741,2.818116,3.282261,1243.333333
Latitude,20640.0,35.631861,2.135952,32.540000,33.930000,34.260000,37.710000,41.950000
Longitude,20640.0,-119.569704,2.003532,-124.350000,-121.800000,-118.490000,-118.010000,-114.310000
MedHouseVal,20640.0,2.068558,1.153956,0.149990,1.196000,1.797000,2.647250,5.000010


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error

X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
print("Baseline RMSE:", root_mean_squared_error(y_test, baseline.predict(X_test)))


Baseline RMSE: 1.1448563543099792


In [8]:
print("Missing Values:", X_train.isna().sum())
print(X_train.skew().sort_values(ascending=False))

skewed = ["AveRooms", "AveBedrms", "Population", "AveOccup"]
display(X_train[skewed].describe(percentiles=[.01, .99]).T)

Missing Values: MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64
AveOccup      88.052663
AveBedrms     23.170746
AveRooms      18.607090
Population     5.276131
MedInc         1.634090
Latitude       0.461504
HouseAge       0.063453
Longitude     -0.288418
dtype: float64


,count,mean,std,min,1%,50%,99%,max
AveRooms,16512.0,5.435235,2.387375,0.888889,2.578195,5.235874,10.318432,141.909091
AveBedrms,16512.0,1.096685,0.433215,0.333333,0.871161,1.049286,2.116536,25.636364
Population,16512.0,1426.453004,1137.056380,3.000000,88.110000,1167.000000,5798.720000,35682.000000
AveOccup,16512.0,3.096961,11.578744,0.692308,1.538838,2.817240,5.410157,1243.333333


In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import Ridge

others = [c for c in X_train.columns if c not in skewed]

log_scale = Pipeline([
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler())
])

preprocess = ColumnTransformer([
    ("skewed", log_scale, skewed),
    ("rest", StandardScaler(), others)
])

ridge = Pipeline([
    ("prep", preprocess), ("model", Ridge(alpha=1.0))
    ])

ridge.fit(X_train, y_train)
print("Ridge MSE:", root_mean_squared_error(y_test, ridge.predict(X_test)))


Ridge MSE: 0.6832665000927511


In [10]:
def add_features(X):
    X = X.copy()
    X["BedroomRatio"] = X["AveBedrms"] / X["AveRooms"]
    X["RoomsPerPerson"] = X["AveRooms"] / X["AveOccup"]
    X["DistSF"] = np.hypot(X["Latitude"] - 37.77, X["Longitude"] + 122.42)
    X["DistLA"] = np.hypot(X["Latitude"] - 34.05, X["Longitude"] + 118.24)
    return X

new_feats = ["BedroomRatio", "RoomsPerPerson", "DistSF", "DistLA"]

preprocess_fe = ColumnTransformer([
    ("skewed", log_scale, skewed),
    ("rest", StandardScaler(), others + new_feats)
])

ridge_fe = Pipeline([
    ("fe", FunctionTransformer(add_features)),
    ("prep", preprocess_fe),
    ("model", Ridge(alpha=1.0))
])

ridge_fe.fit(X_train, y_train)
print("Ridge + FE RMSE:", root_mean_squared_error(y_test, ridge_fe.predict(X_test)))

Ridge + FE RMSE: 0.6773643368885527


In [14]:
import pandas as pd
from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

def make_pipe(model):
    return Pipeline([
        ("fe", FunctionTransformer(add_features)),
        ("prep", preprocess_fe),
        ("model", model)
    ])

models = {
    "ridge": Ridge(alpha=1.0),
    "lasso": Lasso(alpha=0.001),
    "rf": RandomForestRegressor(n_estimators=200, random_state=42),
    "hgb": HistGradientBoostingRegressor(random_state=42)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, m in models.items():
    res = cross_validate(
        make_pipe(m), X_train, y_train, cv=cv,
        scoring="neg_root_mean_squared_error", n_jobs=-1
    )
    rmse = -res["test_score"]
    rows.append({"model":name, "rmse_mean":rmse.mean(), "rmse_std": rmse.std()})


display(pd.DataFrame(rows).sort_values("rmse_mean"))


,model,rmse_mean,rmse_std
3,hgb,0.461610,0.009156
2,rf,0.491955,0.011402
0,ridge,0.665840,0.016817
1,lasso,0.665982,0.017053
